# Part A

In [1]:
import os
import torch
import jiwer
import librosa
import pandas as pd
from transformers import pipeline
from tqdm import tqdm

def normalize_text(text):
    transform = jiwer.Compose([
        jiwer.ToLowerCase(),
        jiwer.RemovePunctuation(),
        jiwer.RemoveMultipleSpaces(),
        jiwer.Strip(),
    ])
    return transform(str(text))

def main():
    tsv_path = "../voice_dataset/az/test.tsv"
    clips_dir = "../voice_dataset/az/clips/"

    df = pd.read_csv(tsv_path, sep='\t')
    
    text_col = "text" if "text" in df.columns else "sentence"

    # complete file path
    df['full_audio_path'] = df['path'].apply(lambda x: os.path.join(clips_dir, x))

    # check if gpu exist
    device = "cuda:0" if torch.cuda.is_available() else "cpu"

    asr_pipeline = pipeline(
        "automatic-speech-recognition",
        model="openai/whisper-small",
        torch_dtype=torch.float32,
        device=device,
        model_kwargs={"attn_implementation": "sdpa"} if torch.cuda.is_available() else {}
    )

    # tell the model to use azerbaijani langauge and not to translate to english
    asr_pipeline.model.config.forced_decoder_ids = asr_pipeline.tokenizer.get_decoder_prompt_ids(language="az", task="transcribe")

    results = []

    # tqdm is just for seeing the progress
    for index, row in tqdm(df.iterrows(), total=len(df), desc="Transcribing Test Split"):
        reference_raw = str(row[text_col])
        audio_file = row["full_audio_path"]

        try:
            audio_array, _ = librosa.load(audio_file, sr=16000)
        except Exception:
            continue

        prediction_raw = asr_pipeline(audio_array)["text"]

        reference_clean = normalize_text(reference_raw)
        prediction_clean = normalize_text(prediction_raw)

        if len(reference_clean) == 0:
            continue

        wer = jiwer.wer(reference_clean, prediction_clean)
        cer = jiwer.cer(reference_clean, prediction_clean)

        results.append({
            "Reference": reference_raw,
            "Prediction": prediction_raw,
            "WER": wer,
            "CER": cer
        })

    results_df = pd.DataFrame(results)

    avg_wer = results_df["WER"].mean() * 100
    avg_cer = results_df["CER"].mean() * 100

    # get the best and worst cases
    df_sorted = results_df.sort_values(by="WER")
    best_5 = df_sorted.head(5)
    worst_5 = df_sorted.tail(5)

    # printing
    print("\n--- ASR Performance Metrics ---")
    print(f"Total Test Clips Evaluated: {len(results_df)}")
    print(f"Average WER: {avg_wer:.2f}%")
    print(f"Average CER: {avg_cer:.2f}%\n")

    print("--- Top 5 Best Predictions ---")
    print(best_5.to_string(index=False))
    print("\n--- Top 5 Worst Predictions ---")
    print(worst_5.to_string(index=False))

if __name__ == "__main__":
    main()

/home/ra3n24/miniconda3/envs/az-asr/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Transcribing Test Split:   0%|          | 0/126 [00:00<?, ?it/s][transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] A custom logits processor of type <class 'transf


--- ASR Performance Metrics ---
Total Test Clips Evaluated: 126
Average WER: 92.77%
Average CER: 42.51%

--- Top 5 Best Predictions ---
                                                    Reference                                                     Prediction      WER      CER
Hal hazırda Xəzər kanalında prodüser olaraq işinə davam edir.  Hal hazırda xəzər kanalında producer olaraq işinə davam edir. 0.111111 0.033333
     Şəhər alim və ixtiraçı Nikola Teslanın doğulduğu yerdir.      Şəhər Alim və İxtiratlı Nikola Teslanın doğulduğu yerdir. 0.125000 0.054545
           Təhsilini davam etdirmək istəsə də mümkün olmayıb.            Təhsilini, davam etdirməc istəsə də mümkün olmayıb. 0.142857 0.020408
  Müxtəlif illərdə fərqli sahələrdə iş fəaliyyətinə başlayır.   Müxtəlif illərdə fərqili sahələrdə iş fəaliyyətinə başlayır. 0.142857 0.017241
                        Şaxtalı aylarda sahil ərazilər donur.                          Şəxdalı aylarda sahil ərazilər donur. 0.200000 0.055556

--- 